In [1]:
import numpy
import pandas
import random
from typing import Any
from sklearn.model_selection import train_test_split

In [2]:
# Define random seeds for reproducibility
SEED: int = 42
random.seed(SEED)
numpy.random.seed(SEED)

In [3]:

def map_icd9_codes(icd_val: Any) -> str:
    """
    The ICD-9 codes are mapped to standard medical categories.
    """
    # If missing or '?' value
    if pandas.isna(icd_val) or str(icd_val).strip() == '?':
        return 'Missing'

    val: str = str(icd_val).strip()

    # 'V' and 'E' codes indicate supplementary or external causes -> 'Other'
    if val.startswith('V') or val.startswith('E'):
        return 'Other'

    try:
        # Convert to float for range checks
        num_val: float = float(val)

        if 390 <= num_val <= 459 or numpy.floor(num_val) == 785:
            return 'Circulatory'    # Circulatory system
        elif 460 <= num_val <= 519 or numpy.floor(num_val) == 786:
            return 'Respiratory'    # Respiratory system
        elif 520 <= num_val <= 579 or numpy.floor(num_val) == 787:
            return 'Digestive'      # Digestive system
        elif numpy.floor(num_val) == 250:
            return 'Diabetes'       # Diabetes
        elif 800 <= num_val <= 999:
            return 'Injury'         # Injury / Accident
        elif 710 <= num_val <= 739:
            return 'Musculoskeletal'# Musculoskeletal system
        elif 580 <= num_val <= 629 or numpy.floor(num_val) == 788:
            return 'Genitourinary'  # Urogenitális
        elif 140 <= num_val <= 239:
            return 'Neoplasms'      # Neoplasms
        else:
            return 'Other'          # Other diseases
    except ValueError:
        return 'Other'

In [4]:
# Diabetes 130-US Hospitals for Years 1999-2008
# https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008
df: pandas.DataFrame = pandas.read_csv("D:\\Downloads\\diabetes_130\\diabetic_data.csv")

print("Raw number of features:", len(df.columns) - 1)
print("Original columns:", df.columns)

Raw number of features: 49
Original columns: Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='object')


In [5]:
# Target variable transformation
# The target variable is binary: 1 if the patient was readmitted within 30 days, 0 otherwise.
df['target_readmitted'] = (df['readmitted'] == '<30').astype(int)

# Time-based (sequential) sorting by encounter_id
df: pandas.DataFrame = df.sort_values(by='encounter_id').reset_index(drop=True)

# Drop highly missing columns
# 'weight': ~97% missing, 'payer_code': ~40% missing, 'medical_specialty': ~49% missing
cols_to_drop: list[str] = ['weight', 'payer_code', 'medical_specialty', 'readmitted']
df: pandas.DataFrame = df.drop(columns=cols_to_drop, errors='ignore')

# Replace '?' values with NaN and categorize or drop minor missing values
# For minor missing values (e.g., race, gender), drop NaN rows or categorize them as 'Unknown'.
# Here, we put them in 'Unknown' to avoid losing data.
df['race'] = df['race'].fillna('Unknown')
df['gender'] = df['gender'].fillna('Unknown')

# Categorize or drop minor missing values
# For minor missing values (e.g., race, gender), drop NaN rows or categorize them as 'Unknown'.
# Here, we put them in 'Unknown' to avoid losing data.
for diag_col in ['diag_1', 'diag_2', 'diag_3']:
    df[diag_col]: pandas.Series = df[diag_col].apply(map_icd9_codes)

# Aggregate previous hospital stays (Optional Feature Engineering, but very useful)
# It's worth summarizing the number of previous visits in a variable
df['total_previous_visits'] = df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']

# Remove identifiers from the model input
# patient_nbr and encounter_id are not predictive variables, but they are useful to keep in the index
df: pandas.DataFrame = df.set_index(['encounter_id', 'patient_nbr'])

# One-Hot encode categorical variables across the ENTIRE dataset
# The target_readmitted (since it's int) remains unchanged!
df: pandas.DataFrame = pandas.get_dummies(df, drop_first=True)

In [6]:
# Find columns with only 1 (or 0) unique values
constant_columns: list[str] = [col for col in df.columns if df[col].nunique() <= 1]
print(f"Constant columns found: {len(constant_columns)}")

if len(constant_columns) > 0:
    print(f"Columns to be dropped: {constant_columns}")
    df: pandas.DataFrame = df.drop(columns=constant_columns)

df_nans: int = df.isna().sum().sum()
print(f"Missing values: {df_nans}")
df: pandas.DataFrame = df.dropna()

df_dupes: int = df.duplicated().sum()
print(f"Duplicated values: {df_dupes}")
df: pandas.DataFrame = df.drop_duplicates()

# Sequential split (based on time sequence) at DataFrame level
# shuffle=False ensures that the first 70% is the earlier period, and the rest is the test
train_df, test_df = train_test_split(
    df,
    test_size=0.3,
    shuffle=False,
    random_state=SEED,
)

train_target_ratio: float = train_df['target_readmitted'].mean() * 100
test_target_ratio: float = test_df['target_readmitted'].mean() * 100
print("Target variable: <30-day readmission rate")
print(f"Training set: {train_target_ratio:.2f}% positive case")
print(f"Test set:     {test_target_ratio:.2f}% positive case")

Constant columns found: 0
Missing values: 0
Duplicated values: 2
Target variable: <30-day readmission rate
Training set: 11.35% positive case
Test set:     10.73% positive case


In [7]:
# Save preprocessed datasets
train_df.to_csv("readmit_130_hospitals_preprocessed_train_data.csv", index=False)
test_df.to_csv("readmit_130_hospitals_preprocessed_test_data.csv", index=False)